In [1]:
from drive_operations import find_duplicates, delete_files
import pandas as pd
from unidecode import unidecode
import duckdb

# Lecture de ma base de données
Tout ce qui est contenu sur mon drive

In [2]:
# Convertir la liste de fichiers en DataFrame
df = pd.read_csv(
    "../data/files.csv",
    delimiter=";",
    encoding="utf-8",
)

In [ ]:
df.tail()

# Est ce qu'il y a une différence entre les fichiers partagés et ce qui n'ont pas de parents?

In [ ]:
df[df['parents'].isna()]

Je pense qu'ici il s'agit des fichiers partagées. Après un peu de picking, je l'ai compris en regardant directement sur mon drive

# Et s'il n'y a pas de parents, ça marche comment?
Je sais que Organisation Rando est un dossier à la base de mon drive. Donc en reprenant l'id du parent

In [ ]:
df[df['id'] == "0AIFpqg8Pz0O0Uk9PVA"]

En clair il n'y a pas de ligne quand il n'y a plus de parents

# Comment ça marche pour un deroulement complet de l'arborescence?
## Exemple avec un fichier java

In [ ]:
df[df['name'] == "NumberOfMessages.java"]

## Que contient le dossier parents pour le fichier issu de la dernière ligne?

In [ ]:
df[df['parents'] == "['1Axsk5N0EN4SJPa5eE4zi48cBB_zmB0-C']"]

## Quel est le parent?

In [ ]:
df[df['id'] == "1Axsk5N0EN4SJPa5eE4zi48cBB_zmB0-C"]

## Que contient le parent du parent?

In [ ]:
df[df['parents'] == "['12W8vXDybQBCYHbp3HxZr48mOrmOKounU']"]

## Quel est le parent du parent?

In [ ]:
df[df['id'] == "12W8vXDybQBCYHbp3HxZr48mOrmOKounU"]

## Etc pour arriver finalement à la racine c'est à dire mon disque

In [ ]:
df[df['id'] == "1Bj0BwP5NrJcmXtbbIy2CkHbq0FQgID90"]

In [ ]:
df[df['id'] == "1KjVJvEw6TgnFco4fsUDUGIvh4AnWyBI_"]

In [ ]:
df[df['name'] == "ENSEM"]

In [ ]:
df[df['id'] == "0AIFpqg8Pz0O0Uk9PVA"]

# Test de dedoublonnage sur les dossiers
Filtrage pour n'avoir que les dossiers et sans les éléments partagés

In [ ]:
df_folds = df[(df['mimeType'] == 'application/vnd.google-apps.folder') & (df['parents'].isna() == False)]

In [ ]:
df_folds

In [ ]:
query= """
SELECT
  SUBSTR(parents, 3, LENGTH(parents) - 4) AS parent_id,
  LIST(name) AS names_list,
  LIST(id) AS ids_list,
  SUM(
    CASE WHEN size THEN size
    ELSE 0
    END
  ) AS total_size,
  COUNT(id) AS count_files,
FROM df
WHERE
  parent_id IS NOT NULL
  -- AND mimeType == 'application/vnd.google-apps.folder'
GROUP BY parent_id
ORDER BY total_size DESC
"""

df_test_1 = duckdb.sql(query).df()

In [ ]:
df_test_1.drop_duplicates(subset=['names_list'], keep='first', inplace=True)

In [ ]:
query = """WITH folder_contents AS (
  SELECT
  SUBSTR(parents, 3, LENGTH(parents) - 4) AS parent_id,
  LIST(name) AS names_list,
  LIST(id) AS ids_list,
  SUM(
    CASE
      WHEN size THEN size
    ELSE 0
    END
  ) AS total_size,
  COUNT(id) AS count_files,
FROM df
WHERE
  parent_id IS NOT NULL
  -- AND mimeType == 'application/vnd.google-apps.folder'
GROUP BY parent_id
)

SELECT
  a.parent_id AS parent_id,
  b.parent_id AS parent_id_to_compare,
  c.name as parent_name_of_a,
  a.names_list AS names_list,
  b.names_list AS name_to_compare,
  a.ids_list AS ids_list,
  b.ids_list AS ids_list_to_compare,
  a.total_size AS total_size,
  b.total_size AS total_size_to_compare,
FROM
  folder_contents a
INNER JOIN
  folder_contents b
ON
  a.names_list = b.names_list
  and a.parent_id != b.parent_id
LEFT JOIN df c
ON
  a.parent_id = c.id
ORDER BY total_size DESC
"""
df_test_2 = duckdb.sql(query).df()

In [ ]:
df_test_2.head(15)

In [ ]:
query = """WITH folder_contents AS (
  SELECT
  SUBSTR(parents, 3, LENGTH(parents) - 4) AS parent_id,
  LIST(name) AS names_list,
  LIST(id) AS ids_list,
  SUM(
    CASE
      WHEN size THEN size
      ELSE 0
    END
  ) AS total_size,
  COUNT(id) AS count_files,
FROM df
WHERE
  parent_id IS NOT NULL
  -- AND mimeType == 'application/vnd.google-apps.folder'
GROUP BY parent_id
),

tree AS (
SELECT
  a.name AS name,
  a.id as id,
  b.name AS parent_name,
  SUBSTR(a.parents, 3, LENGTH(a.parents) - 4) AS parent_id,
  SUBSTR(b.parents, 3, LENGTH(b.parents) - 4) AS grand_parent_id,
FROM df a
INNER JOIN df b
ON SUBSTR(a.parents, 3, LENGTH(a.parents) - 4) = b.id
WHERE
  a.parents IS NOT NULL
  AND b.parents IS NOT NULL
)

SELECT
  t.parent_id AS parent_id,
  t.grand_parent_id AS grand_parent_id,
  t.name AS name,
  t.parent_name AS parent_name,
  f.names_list AS names_list,
  f.ids_list AS ids_list,
  f.total_size AS total_size,
  f.count_files AS count_files,
FROM tree t
INNER JOIN folder_contents f
  ON f.parent_id = t.parent_id
WHERE
  t.parent_id IS NOT NULL
  AND t.grand_parent_id IS NOT NULL
ORDER BY f.total_size DESC
"""
df_test_3 = duckdb.sql(query).df()

#   SUBSTR(d.parents, 3, LENGTH(d.parents) - 4) AS parent_id,
#   d.name,
#   f.names_list AS names_list,
#   -- f.names_list || LIST(d.name) AS names_list_concat,
#   f.ids_list,
#   f.total_size,
#   f.count_files
# FROM df d
# INNER JOIN folder_contents f
#   ON f.parent_id = d.id
# WHERE
#   d.mimeType = 'application/vnd.google-apps.folder'
#   AND d.parents IS NOT NULL
# ORDER BY f.total_size DESC

In [ ]:
df_test_3.head(10)

In [ ]:
df_test_3.loc[0,"names_list"]

In [6]:
query = """WITH RECURSIVE folder_tree AS (
  SELECT
    -- Extraire l'ID du parent à partir de la chaîne de caractères
    SUBSTR(parents, 3, LENGTH(parents) - 4) AS parent_id,
    
    -- Identifiant du fichier
    id,
    
    -- Nom du fichier
    name,
    
    -- Concatenation des noms de fichiers / dossiers contenu par le parent
    ARRAY_AGG(name) OVER(
      PARTITION BY parents
    ) AS names_array,
    
    -- Concaténation des IDs de fichiers / dossiers contenu par le parent
    ARRAY_AGG(id) OVER(
      PARTITION BY parents
    ) AS ids_array,
    
    -- Somme de la taille des fichiers contenu par le parent
    SUM(COALESCE(size, 0)) OVER(
      PARTITION BY parents
    ) AS total_size,
    
    -- Compte le nombre de fichiers contenu par le parent
    COUNT(id) OVER(
      PARTITION BY parents
    ) AS count_files_folders
  FROM df
  WHERE parent_id IS NOT NULL

  UNION ALL

  SELECT
    -- ID du parent
    children_ft.parent_id,
    
    -- Identifiant du fichier / dossier
    children_ft.id,

    -- Nom du fichier / dossier
    children_ft.name,

    -- Concaténation des noms de fichiers / dossiers contenu par le parent
    parent_ft.names_array || children_ft.names_array AS names_array,

    -- Concaténation des IDs de fichiers / dossiers contenu par le parent
    parent_ft.ids_array || children_ft.ids_array AS ids_array,

    -- Somme de la taille des fichiers contenu par le parent
    parent_ft.total_size + COALESCE(children_ft.total_size, 0) AS total_size,

    -- Compte le nombre de fichiers contenu par le parent
    parent_ft.count_files_folders + children_ft.count_files_folders AS count_files_folders,
    
  -- Récursivement, on continue à explorer les parents
  FROM
    folder_tree parent_ft
  INNER JOIN
    folder_tree children_ft
  ON
    parent_ft.id = children_ft.parent_id
)

-- Sélection finale : trouver les dossiers ayant les mêmes éléments enfants mais des parents différents
SELECT
  a.parent_id AS parent_folder1,
  b.parent_id AS parent_folder2,
  a.name AS folder_name,
  a.total_size,
  a.count_files_folders,
  a.names_array
FROM folder_tree a
JOIN folder_tree b
  ON a.names_array = b.names_array
  AND a.parent_id != b.parent_id
ORDER BY a.total_size DESC;
"""
df_test_4 = duckdb.sql(query).df()

In [7]:
df_test_4

,parent_folder1,parent_folder2,folder_name,total_size,count_files_folders,names_array
0,1RsPaGk0kSjMiLBAaSqEO8Me-qYqqQYre,1K95eA08Zed0SebOlyQZJMgmTeohTFNjU,PerfReport.html,6.088608e+09,12828,"[Expériences professionnelles, Apprentissage p..."
1,1sHbmHMwS6wchDNa6AvuddVC9-vv5kMiu,1K95eA08Zed0SebOlyQZJMgmTeohTFNjU,PerfReport.html,6.088608e+09,12828,"[Expériences professionnelles, Apprentissage p..."
2,1VlNW3QEpSb2tkMrHuDcsV_5ClXMynTtz,1K95eA08Zed0SebOlyQZJMgmTeohTFNjU,PerfReport.html,6.088608e+09,12828,"[Expériences professionnelles, Apprentissage p..."
3,1RsPaGk0kSjMiLBAaSqEO8Me-qYqqQYre,1K95eA08Zed0SebOlyQZJMgmTeohTFNjU,logfiles,6.088608e+09,12828,"[Expériences professionnelles, Apprentissage p..."
4,1sHbmHMwS6wchDNa6AvuddVC9-vv5kMiu,1K95eA08Zed0SebOlyQZJMgmTeohTFNjU,logfiles,6.088608e+09,12828,"[Expériences professionnelles, Apprentissage p..."
...,...,...,...,...,...,...
4595,1jrHu9TcOBjv2oWhZB8ElZ8xkuJMjx0u7,1Uux07UDq7_0qDNuj1Wejj5m-d_U2Bdlf,TD,0.000000e+00,118,"[Expériences professionnelles, Apprentissage p..."
4596,1Uux07UDq7_0qDNuj1Wejj5m-d_U2Bdlf,1jrHu9TcOBjv2oWhZB8ElZ8xkuJMjx0u7,Annales,0.000000e+00,118,"[Expériences professionnelles, Apprentissage p..."
4597,1Uux07UDq7_0qDNuj1Wejj5m-d_U2Bdlf,1jrHu9TcOBjv2oWhZB8ElZ8xkuJMjx0u7,TD,0.000000e+00,118,"[Expériences professionnelles, Apprentissage p..."
4598,1jrHu9TcOBjv2oWhZB8ElZ8xkuJMjx0u7,1Uux07UDq7_0qDNuj1Wejj5m-d_U2Bdlf,Annales,0.000000e+00,118,"[Expériences professionnelles, Apprentissage p..."


In [4]:
df_test_4[df_test_4["name"].isin(["2016", "2019", "2018", "2017", "2015", "2014"])]

KeyError: 'name'

In [ ]:
df_test_4[df_test_4["parent_id"] == "1at09_zMtj7M8j3-pDkoRoWFDDAQt-O47"]

In [ ]:
df_test_4[df_test_4["id"] == "1at09_zMtj7M8j3-pDkoRoWFDDAQt-O47"]

In [ ]:
display(df_test_4[df_test_4["id"] == "1-9bQdI7dsJruWw4oV4x8tmmiQTeDbPDL"])
display(df_test_4[df_test_4["id"] == "1Q0cT7mQeHOCvHoistZj2yRZRFbwPPia9"])

In [ ]:
df_test_4.loc[7604,"names_array"]

In [ ]:
query = """WITH RECURSIVE folder_contents AS (
  SELECT
    SUBSTR(parents, 3, LENGTH(parents) - 4) AS parent_id,
    ARRAY_AGG(name) AS names_array,
    ARRAY_AGG(id) AS ids_array,
    SUM(COALESCE(size, 0)) AS total_size,
    COUNT(id) AS count_files_folders
  FROM df
  WHERE parents IS NOT NULL
  GROUP BY parent_id

  UNION ALL

  SELECT
    parent_ft.parent_id,
    ARRAY(SELECT DISTINCT name FROM UNNEST(parent_ft.names_array || ARRAY[d.name])) AS names_array,
    ARRAY(SELECT DISTINCT id FROM UNNEST(parent_ft.ids_array || ARRAY[d.id])) AS ids_array,
    parent_ft.total_size + COALESCE(d.size, 0) AS total_size,
    parent_ft.count_files_folders + 1 AS count_files_folders
  FROM folder_contents parent_ft
  INNER JOIN df d
  ON parent_ft.parent_id = SUBSTR(d.parents, 3, LENGTH(d.parents) - 4)
)

SELECT * FROM folder_contents
ORDER BY total_size DESC;

"""

df_test_5 = duckdb.sql(query).df()

In [ ]:
df_test_5

In [ ]:
df[df["id"] == "1vWwmK4eLzX_yb2yxz3IVtMqfCqZ9Lw9l"]

# Filtrage pour enlever les dossiers
C'est tout à fait normal que les dossiers soit en doublons si des arborescences se ressemble

In [ ]:
df_folds = df[df['mimeType'] == 'application/vnd.google-apps.folder']

In [ ]:
df_folds.groupby('name')['name'].count().reset_index(name='count').query('count < 6 & count > 1')

In [ ]:
df = df[df['mimeType'] != 'application/vnd.google-apps.folder']

# Est-ce qu'il y a des doublous ?
Test simple sur le noms des fichiers

In [ ]:
# Regex constants
DUPLICATE_PATTERN = r'\(\d{1,2}\)|\bcopie\s*de\b'
EXTENSION_CLEANUP_PATTERN = r'\s+((\.\w+)+)$'
EXTENSION_CLEANUP_REPLACEMENT = r'\1'

In [ ]:
# Normalisation du nom
df['normalized_name'] = df['name'].fillna('').apply(unidecode)

# Création du base_name pour tout le monde
df['base_name'] = (
    df['normalized_name']
    .str.replace(DUPLICATE_PATTERN, '', regex=True)
    .str.replace(EXTENSION_CLEANUP_PATTERN, EXTENSION_CLEANUP_REPLACEMENT, regex=True)
    .str.strip()
    .str.lower()
)

In [ ]:
def mark_duplicates(folder):
    # Ceux que l’on garde dans ce dossier
    keep = folder.sort_values(by='modifiedTime').drop_duplicates(subset='base_name', keep='last')
    # Ceuxw que l’on supprime dans ce dossier
    folder['to_remove'] = ~folder.index.isin(keep.index)
    return folder

In [ ]:
# df = df.groupby('parents', group_keys=False).apply(mark_duplicates)

In [ ]:
df[df['base_name'] == 'faceavant.png']
df[df['base_name'] == "TPNRJ1A etn 4 Cld_31(1).pdf"]